In [ ]:
import copy
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import sys
import scrapbook as sb
from statsmodels.discrete.conditional_models import ConditionalLogit

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [ ]:
seed = 1
sampling_rate = 10
topple_delta_s = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 128
patience = 15
encoder_hidden_dim1 = 128
encoder_hidden_dim2 = 64
encoding_dim = 32
decoder_hidden_dim1 = 64
decoder_hidden_dim2 = 128
lr=0.001
n_splits = 5

In [ ]:
rng = np.random.RandomState(seed)

In [ ]:
data = pd.read_parquet("../data/SRV3490.parquet")
data

In [ ]:
accidents = data["crash_ride_id"].unique()
accidents

In [ ]:
# truncate all rides to topple_delta_s before first topple
# accidents use their own ride_id; baselines use crash_ride_id (linked accident)
first_topple_idx = data[data["topple"] == 1].groupby("ride_id")["time_index"].min()
topple_delta_idx = int(topple_delta_s * sampling_rate)
cutoff_map = (first_topple_idx - topple_delta_idx).to_dict()

lookup_id = data["crash_ride_id"].fillna(data["ride_id"])
data = data[data["time_index"] < lookup_id.map(cutoff_map)]
data

In [ ]:
ride_ids = data.groupby("ride_id").first().index.values
ride_ids

In [ ]:
labels = pd.DataFrame({
    'ride_id': ride_ids
})
labels['label'] = labels['ride_id'].isin(accidents).map({True: 'accident', False: 'normal'})
labels['crash_group'] = labels['ride_id'].map(
    data.groupby("ride_id")["crash_ride_id"].first()
).fillna(labels['ride_id'])
labels['label'].value_counts()

In [ ]:
features = ["ax", "ay", "az", "rx", "ry", "rz"]

data_np = reshape_to_numpy(
    data,
    features = features
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

In [ ]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

In [ ]:
# Define Autoencoder architecture with Lightning
class Autoencoder(L.LightningModule):
    def __init__(
        self,
        input_dim,
        encoding_dim,
        encoder_hidden_dim1,
        encoder_hidden_dim2,
        decoder_hidden_dim1,
        decoder_hidden_dim2,
        lr,
    ):
        super().__init__()
        self.save_hyperparameters()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim1, encoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim2, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, decoder_hidden_dim1),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim1, decoder_hidden_dim2),
            nn.ReLU(),
            nn.Linear(decoder_hidden_dim2, input_dim)
        )
        self.criterion = nn.MSELoss()
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)
    
    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_hat = self.forward(x)
        loss = self.criterion(x_hat, x)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
# 5-fold group CV with AE
y_true = (labels['label'] == 'accident').astype(int).values
groups = labels['crash_group'].values
gkf = GroupKFold(n_splits=n_splits)
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_feat, y_true, groups)):
    fold_scaler = StandardScaler()
    X_train = fold_scaler.fit_transform(X_feat[train_idx])
    X_test = fold_scaler.transform(X_feat[test_idx])

    pca = PCA(n_components=encoding_dim).fit(X_train)
    print(f"Variance explained (train): {sum(pca.explained_variance_ratio_):.4f}")

    L.seed_everything(rng.randint(1000))
    autoencoder = Autoencoder(
        input_dim=X_train.shape[1],
        encoding_dim=encoding_dim,
        encoder_hidden_dim1=encoder_hidden_dim1,
        encoder_hidden_dim2=encoder_hidden_dim2,
        decoder_hidden_dim1=decoder_hidden_dim1,
        decoder_hidden_dim2=decoder_hidden_dim2,
        lr=lr,
    )

    X_train_tensor = torch.FloatTensor(X_train)
    dataset = TensorDataset(X_train_tensor, X_train_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    state = {'best_ewm_loss': float('inf'), 'best_state': None, 'ewm': None}

    class FoldLossTracker(L.pytorch.callbacks.Callback):
        def on_train_epoch_end(self, trainer, pl_module):
            loss = float(trainer.callback_metrics['train_loss'])
            if state['ewm'] is None:
                state['ewm'] = loss
            else:
                state['ewm'] = 0.2 * loss + 0.8 * state['ewm']
            if state['ewm'] < state['best_ewm_loss']:
                state['best_ewm_loss'] = state['ewm']
                state['best_state'] = copy.deepcopy(pl_module.state_dict())
            pl_module.log('train_loss_ewm_avg', state['ewm'])

    trainer = L.Trainer(
        max_epochs=300, accelerator='auto', devices=1,
        callbacks=[
            EarlyStopping(monitor='train_loss_ewm_avg', patience=patience,
                          verbose=False, mode='min', check_on_train_epoch_end=True),
            FoldLossTracker(),
        ],
        enable_progress_bar=False,
    )
    trainer.fit(autoencoder, dataloader)

    if state['best_state'] is not None:
        autoencoder.load_state_dict(state['best_state'])

    autoencoder.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        Z_train = autoencoder.encode(X_train_tensor).numpy()
        Z_test = autoencoder.encode(X_test_tensor).numpy()

    enc_scaler = StandardScaler()
    Z_train = enc_scaler.fit_transform(Z_train)
    Z_test = enc_scaler.transform(Z_test)

    lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
    lof.fit(Z_train)
    anomaly_scores[test_idx] = -lof.score_samples(Z_test)

    print(f"Fold {fold+1}/{n_splits} done")

assert not np.isnan(anomaly_scores).any(), "NaN found in anomaly scores"
labels['anomaly_score'] = anomaly_scores

In [ ]:
# Conditional logistic regression: top quantile indicator → accident, stratified by crash_group
endog = (labels['label'] == 'accident').astype(int)
quantiles = [0.95, 0.99]

results = []
for q in quantiles:
    q_pct = int(q * 100)
    top = (labels['anomaly_score'] >= labels['anomaly_score'].quantile(q)).astype(int)

    clogit = ConditionalLogit(
        endog=endog,
        exog=top,
        groups=labels['crash_group'],
    ).fit(disp=False)

    or_ = float(np.exp(clogit.params.iloc[0]))
    ci = np.exp(clogit.conf_int().iloc[0])
    results.append({'quantile': q_pct, 'or': or_, 'ci_l': float(ci[0]), 'ci_u': float(ci[1]), 'pval': float(clogit.pvalues.iloc[0])})

res = pd.DataFrame(results)

# Glue key quantiles
for q in quantiles:
    q_pct = int(q * 100)
    row = res[res['quantile'] == q_pct].iloc[0]
    sb.glue(f"SRV3490_q{q_pct}_or", row['or'])
    sb.glue(f"SRV3490_q{q_pct}_pval", row['pval'])
    sb.glue(f"SRV3490_q{q_pct}_sig05", int(row['pval'] < 0.05))
    sb.glue(f"SRV3490_q{q_pct}_sig01", int(row['pval'] < 0.01))
    sb.glue(f"SRV3490_q{q_pct}_sig001", int(row['pval'] < 0.001))
res

In [ ]:
# Conditional logistic regression: continuous anomaly score (linear) → accident
endog_cont = (labels['label'] == 'accident').astype(int)

clogit_cont = ConditionalLogit(
    endog=endog_cont,
    exog=labels[['anomaly_score']] / labels['anomaly_score'].std(),
    groups=labels['crash_group'],
).fit(disp=False)

or_cont = float(np.exp(clogit_cont.params.iloc[0]))
ci_cont = np.exp(clogit_cont.conf_int().iloc[0])
pval_cont = float(clogit_cont.pvalues.iloc[0])

print(f"OR per 1-SD = {or_cont:.4f} [{ci_cont[0]:.4f}, {ci_cont[1]:.4f}], p = {pval_cont:.4g}")
sb.glue("SRV3490_cont_or", or_cont)
sb.glue("SRV3490_cont_pval", pval_cont)
sb.glue("SRV3490_cont_sig05", int(pval_cont < 0.05))
sb.glue("SRV3490_cont_sig01", int(pval_cont < 0.01))
sb.glue("SRV3490_cont_sig001", int(pval_cont < 0.001))